# 03 - Análise Social: Renda por Bairro em Recife

Agrega os setores censitários por bairro, calcula renda média, % abaixo da linha de pobreza e gera mapa coroplético interativo.

In [15]:
import os
import json
import pandas as pd
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [16]:
ROOT        = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
INPUT_PATH  = os.path.join(ROOT, 'data', 'processed', 'recife_renda.geojson')
OUTPUT_CSV  = os.path.join(ROOT, 'data', 'processed', 'recife_renda_bairros.csv')

print(f'Entrada : {INPUT_PATH}')
print(f'Saída   : {OUTPUT_CSV}')

Entrada : c:\Users\rodri\OneDrive\Documents\ProjetosPessoais\IVU-RECIFE\data\processed\recife_renda.geojson
Saída   : c:\Users\rodri\OneDrive\Documents\ProjetosPessoais\IVU-RECIFE\data\processed\recife_renda_bairros.csv


In [17]:
# Carrega o GeoJSON com setores censitários e dados de renda
gdf = gpd.read_file(INPUT_PATH)

COLS_NUM = ['V06001', 'V06002', 'V06003', 'V06004', 'V06005', 'V06006']
for col in COLS_NUM:
    gdf[col] = pd.to_numeric(gdf[col], errors='coerce')

print(f'Setores carregados  : {len(gdf)}')
print(f'Bairros únicos      : {gdf["NM_BAIRRO"].nunique()}')
print()
# Dicionário das variáveis utilizadas
print('Variáveis:')
print('  V06001 – responsáveis por domicílio (total do setor)')
print('  V06003 – média de moradores por domicílio')
print('  V06004 – rendimento médio mensal do responsável (R$)')
print('  V06006 – rendimento mediano mensal do responsável (R$)')

display(gdf[['CD_SETOR', 'NM_BAIRRO', 'V06001', 'V06003', 'V06004', 'V06006']].head())

Setores carregados  : 2835
Bairros únicos      : 94

Variáveis:
  V06001 – responsáveis por domicílio (total do setor)
  V06003 – média de moradores por domicílio
  V06004 – rendimento médio mensal do responsável (R$)
  V06006 – rendimento mediano mensal do responsável (R$)


,CD_SETOR,NM_BAIRRO,V06001,V06003,V06004,V06006
0,261160605180001,Boa Vista,346.0,1.53,5054.18,3100.0
1,261160605180002,Boa Vista,220.0,1.75,5567.78,4500.0
2,261160605180003,Boa Vista,237.0,1.25,3006.58,2300.0
3,261160605180004,Boa Vista,244.0,1.21,4081.47,3000.0
4,261160605180005,Boa Vista,292.0,1.03,3194.82,2400.0


## 1 – Renda média por bairro

Média ponderada de `V06004` (renda média do responsável) usando `V06001` (total de responsáveis do setor) como peso.

In [18]:
gdf['renda_pond'] = gdf['V06004'] * gdf['V06001']

agg = (
    gdf
    .groupby(['CD_BAIRRO', 'NM_BAIRRO'], dropna=False)
    .agg(
        n_setores          = ('CD_SETOR',    'count'),
        total_responsaveis = ('V06001',      'sum'),
        soma_pond          = ('renda_pond',  'sum'),
        mediana_renda      = ('V06006',      'median'),
    )
    .reset_index()
)

agg['renda_media'] = (agg['soma_pond'] / agg['total_responsaveis']).round(2)
agg = agg.drop(columns=['soma_pond'])

print(f'Total de bairros: {len(agg)}')
display(agg.sort_values('renda_media', ascending=False).head(10))

Total de bairros: 95


,CD_BAIRRO,NM_BAIRRO,n_setores,total_responsaveis,mediana_renda,renda_media
13,2611606014,Jaqueira,3,509.0,15000.0,16337.43
54,2611606055,Casa Forte,10,2325.0,10001.0,14308.91
53,2611606054,Parnamirim,14,2573.0,10000.0,13405.17
56,2611606057,Poço,12,1965.0,9000.0,12168.53
12,2611606013,Graças,37,7943.0,10000.0,11761.22
55,2611606056,Santana,6,1013.0,8500.0,11677.45
57,2611606058,Monteiro,9,2342.0,10001.0,11028.80
14,2611606015,Aflitos,10,1998.0,8000.0,11008.62
16,2611606017,Rosarinho,12,2818.0,7225.0,10288.50
15,2611606016,Espinheiro,19,4374.0,8000.0,9868.23


## 2 – % de pessoas abaixo da linha de pobreza

**Metodologia:** estimativa de renda per capita do setor = `V06004 / V06003` (renda média ÷ moradores/domicílio).  
Setores com renda per capita < R$ 436/mês são classificados como *em situação de pobreza* (critério Bolsa Família 2022).  
O % por bairro é calculado como a fração de responsáveis nesses setores sobre o total do bairro.

In [19]:
LINHA_POBREZA = 436.0  # R$/mês per capita — critério Bolsa Família 2022

gdf['renda_percapita'] = gdf['V06004'] / gdf['V06003']
gdf['em_pobreza']      = gdf['renda_percapita'] < LINHA_POBREZA
gdf['resp_em_pobreza'] = gdf['em_pobreza'].astype(float) * gdf['V06001']

agg_pob = (
    gdf
    .groupby(['CD_BAIRRO', 'NM_BAIRRO'], dropna=False)
    .agg(
        resp_pobreza = ('resp_em_pobreza', 'sum'),
        total_resp   = ('V06001',          'sum'),
    )
    .reset_index()
)

agg_pob['pct_pobreza'] = (agg_pob['resp_pobreza'] / agg_pob['total_resp'] * 100).round(1)

agg = agg.merge(
    agg_pob[['NM_BAIRRO', 'resp_pobreza', 'pct_pobreza']],
    on='NM_BAIRRO',
    how='left'
)

print(f'Linha de pobreza: R$ {LINHA_POBREZA:.0f}/mês per capita')
print(f'Bairros com > 0% em pobreza: {(agg["pct_pobreza"] > 0).sum()}')
display(agg.sort_values('pct_pobreza', ascending=False).head(10))

Linha de pobreza: R$ 436/mês per capita
Bairros com > 0% em pobreza: 50


,CD_BAIRRO,NM_BAIRRO,n_setores,total_responsaveis,mediana_renda,renda_media,resp_pobreza,pct_pobreza
9,2611606010,Cabanga,4,625.0,1225.0,1681.01,313.0,50.1
8,2611606009,Ilha Joana Bezerra,23,4260.0,1200.0,1134.42,1629.0,38.2
30,2611606031,Alto Santa Terezinha,14,2305.0,1212.0,1349.52,643.0,27.9
10,2611606011,São José,21,3154.0,1212.0,2187.88,724.0,23.0
26,2611606027,Passarinho,46,8356.0,1212.0,1193.83,1862.0,22.3
28,2611606029,Linha do Tiro,28,4900.0,1212.0,1426.11,811.0,16.6
19,2611606020,Campina do Barreto,14,3093.0,1212.0,1972.12,475.0,15.4
34,2611606035,Água Fria,66,13482.0,1212.0,1558.77,2059.0,15.3
76,2611606077,Estância,17,2719.0,1212.0,1955.69,415.0,15.3
1,2611606002,Santo Amaro,50,10046.0,1212.0,3591.84,1504.0,15.0


## 3 – Geometria por bairro

Dissolve os polígonos de setores para obter um polígono único por bairro.

In [20]:
geo_bairro = (
    gdf[['NM_BAIRRO', 'geometry']]
    .dissolve(by='NM_BAIRRO')
    .reset_index()
    .to_crs(epsg=4326)
)

geo_bairro = geo_bairro.merge(agg, on='NM_BAIRRO', how='left')

print(f'Polígonos de bairro: {len(geo_bairro)}')
display(geo_bairro[['NM_BAIRRO', 'n_setores', 'total_responsaveis', 'renda_media', 'pct_pobreza']].head())

Polígonos de bairro: 94


,NM_BAIRRO,n_setores,total_responsaveis,renda_media,pct_pobreza
0,Aflitos,10,1998.0,11008.62,0.0
1,Afogados,68,11522.0,1949.84,7.5
2,Alto José Bonifácio,19,3605.0,1455.46,10.0
3,Alto José do Pinho,19,3945.0,1488.98,4.6
4,Alto Santa Terezinha,14,2305.0,1349.52,27.9


## 4 – Mapa coroplético de renda por bairro

In [21]:
geojson = json.loads(geo_bairro.to_json())
for feat in geojson['features']:
    feat['id'] = feat['properties']['NM_BAIRRO']

fig = px.choropleth_mapbox(
    geo_bairro,
    geojson=geojson,
    locations='NM_BAIRRO',
    featureidkey='properties.NM_BAIRRO',
    color='renda_media',
    color_continuous_scale='RdYlGn',
    range_color=[geo_bairro['renda_media'].quantile(0.05), geo_bairro['renda_media'].quantile(0.95)],
    mapbox_style='carto-positron',
    zoom=11,
    center={'lat': -8.05, 'lon': -34.92},
    opacity=0.75,
    hover_name='NM_BAIRRO',
    hover_data={
        'NM_BAIRRO':          False,
        'renda_media':        ':,.0f',
        'mediana_renda':      ':,.0f',
        'pct_pobreza':        ':.1f',
        'total_responsaveis': ':,.0f',
    },
    labels={
        'renda_media':        'Renda Média (R$)',
        'mediana_renda':      'Mediana Renda (R$)',
        'pct_pobreza':        '% em Pobreza',
        'total_responsaveis': 'Total Responsáveis',
    },
    title='Renda Média do Responsável por Bairro — Recife (Censo 2022)',
)

fig.update_layout(
    margin=dict(l=0, r=0, t=50, b=0),
    height=620,
    coloraxis_colorbar=dict(title='Renda Média<br>(R$)'),
)

fig.show(config={'scrollZoom': True})

## 5 – Top 10 bairros mais ricos e mais pobres

In [22]:
COLS_EXIBIR = ['NM_BAIRRO', 'renda_media', 'mediana_renda', 'pct_pobreza', 'total_responsaveis']

top10_ricos  = agg.nlargest(10,  'renda_media')[COLS_EXIBIR].reset_index(drop=True)
top10_pobres = agg.nsmallest(10, 'renda_media')[COLS_EXIBIR].reset_index(drop=True)

top10_ricos.index  += 1
top10_pobres.index += 1

print('TOP 10 — BAIRROS MAIS RICOS (renda média do responsável):')
display(
    top10_ricos.style
    .format({'renda_media': 'R$ {:,.0f}', 'mediana_renda': 'R$ {:,.0f}', 'pct_pobreza': '{:.1f}%', 'total_responsaveis': '{:,.0f}'})
    .background_gradient(subset=['renda_media'], cmap='Greens')
)

print('\nTOP 10 — BAIRROS MAIS POBRES (renda média do responsável):')
display(
    top10_pobres.style
    .format({'renda_media': 'R$ {:,.0f}', 'mediana_renda': 'R$ {:,.0f}', 'pct_pobreza': '{:.1f}%', 'total_responsaveis': '{:,.0f}'})
    .background_gradient(subset=['renda_media'], cmap='Reds_r')
)

TOP 10 — BAIRROS MAIS RICOS (renda média do responsável):


,NM_BAIRRO,renda_media,mediana_renda,pct_pobreza,total_responsaveis
1,Jaqueira,"R$ 16,337","R$ 15,000",0.0%,509
2,Casa Forte,"R$ 14,309","R$ 10,001",0.0%,"2,325"
3,Parnamirim,"R$ 13,405","R$ 10,000",2.4%,"2,573"
4,Poço,"R$ 12,169","R$ 9,000",0.0%,"1,965"
5,Graças,"R$ 11,761","R$ 10,000",0.0%,"7,943"
6,Santana,"R$ 11,677","R$ 8,500",0.0%,"1,013"
7,Monteiro,"R$ 11,029","R$ 10,001",0.0%,"2,342"
8,Aflitos,"R$ 11,009","R$ 8,000",0.0%,"1,998"
9,Rosarinho,"R$ 10,288","R$ 7,225",0.0%,"2,818"
10,Espinheiro,"R$ 9,868","R$ 8,000",0.0%,"4,374"



TOP 10 — BAIRROS MAIS POBRES (renda média do responsável):


,NM_BAIRRO,renda_media,mediana_renda,pct_pobreza,total_responsaveis
1,Recife,"R$ 1,117","R$ 1,212",0.0%,181
2,Ilha Joana Bezerra,"R$ 1,134","R$ 1,200",38.2%,"4,260"
3,Passarinho,"R$ 1,194","R$ 1,212",22.3%,"8,356"
4,Peixinhos,"R$ 1,195","R$ 1,212",10.4%,951
5,Caçote,"R$ 1,246","R$ 1,212",4.4%,"3,182"
6,Alto Santa Terezinha,"R$ 1,350","R$ 1,212",27.9%,"2,305"
7,Córrego do Jenipapo,"R$ 1,364","R$ 1,212",0.0%,"2,854"
8,Dois Unidos,"R$ 1,371","R$ 1,212",1.9%,"11,063"
9,Linha do Tiro,"R$ 1,426","R$ 1,212",16.6%,"4,900"
10,Brejo de Beberibe,"R$ 1,427","R$ 1,212",8.4%,"2,754"


In [23]:
fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Top 10 Mais Ricos', 'Top 10 Mais Pobres'],
    horizontal_spacing=0.25,
)

fig2.add_trace(
    go.Bar(
        x=top10_ricos['renda_media'],
        y=top10_ricos['NM_BAIRRO'],
        orientation='h',
        marker_color='#27ae60',
        name='Mais Ricos',
        text=top10_ricos['renda_media'].apply(lambda x: f'R$ {x:,.0f}'),
        textposition='outside',
    ),
    row=1, col=1,
)

fig2.add_trace(
    go.Bar(
        x=top10_pobres['renda_media'],
        y=top10_pobres['NM_BAIRRO'],
        orientation='h',
        marker_color='#c0392b',
        name='Mais Pobres',
        text=top10_pobres['renda_media'].apply(lambda x: f'R$ {x:,.0f}'),
        textposition='outside',
    ),
    row=1, col=2,
)

fig2.update_yaxes(autorange='reversed')
fig2.update_layout(
    height=420,
    showlegend=False,
    title='Desigualdade de Renda entre Bairros — Recife (Censo 2022)',
    margin=dict(l=10, r=10, t=60, b=10),
)

fig2.show()

## 6 – Salvar resultado em CSV

In [24]:
antes = len(agg)
agg = agg.dropna(subset=['NM_BAIRRO']).reset_index(drop=True)
removidas = antes - len(agg)

print(f'Linhas antes da limpeza : {antes}')
print(f'Linhas removidas (NaN)  : {removidas}')
print(f'Linhas após limpeza     : {len(agg)}')
print()
print('Verificação de NaN no dataframe final:')
display(agg.isnull().sum().to_frame('qtd_nulos'))

Linhas antes da limpeza : 95
Linhas removidas (NaN)  : 1
Linhas após limpeza     : 94

Verificação de NaN no dataframe final:


,qtd_nulos
CD_BAIRRO,0
NM_BAIRRO,0
n_setores,0
total_responsaveis,0
mediana_renda,0
renda_media,0
resp_pobreza,0
pct_pobreza,0


In [25]:
agg_out = agg[[
    'CD_BAIRRO', 'NM_BAIRRO',
    'n_setores', 'total_responsaveis',
    'renda_media', 'mediana_renda',
    'resp_pobreza', 'pct_pobreza',
]].copy()

agg_out.columns = [
    'cd_bairro', 'nm_bairro',
    'n_setores', 'total_responsaveis',
    'renda_media_responsavel', 'mediana_renda_responsavel',
    'responsaveis_em_pobreza', 'pct_abaixo_pobreza',
]

agg_out = agg_out.sort_values('renda_media_responsavel', ascending=False).reset_index(drop=True)

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
agg_out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'Arquivo salvo: {OUTPUT_CSV}')
print(f'Total de bairros: {len(agg_out)}')
print(f'Colunas: {agg_out.columns.tolist()}')
display(agg_out.head(10))

Arquivo salvo: c:\Users\rodri\OneDrive\Documents\ProjetosPessoais\IVU-RECIFE\data\processed\recife_renda_bairros.csv
Total de bairros: 94
Colunas: ['cd_bairro', 'nm_bairro', 'n_setores', 'total_responsaveis', 'renda_media_responsavel', 'mediana_renda_responsavel', 'responsaveis_em_pobreza', 'pct_abaixo_pobreza']


,cd_bairro,nm_bairro,n_setores,total_responsaveis,renda_media_responsavel,mediana_renda_responsavel,responsaveis_em_pobreza,pct_abaixo_pobreza
0,2611606014,Jaqueira,3,509.0,16337.43,15000.0,0.0,0.0
1,2611606055,Casa Forte,10,2325.0,14308.91,10001.0,0.0,0.0
2,2611606054,Parnamirim,14,2573.0,13405.17,10000.0,63.0,2.4
3,2611606057,Poço,12,1965.0,12168.53,9000.0,0.0,0.0
4,2611606013,Graças,37,7943.0,11761.22,10000.0,0.0,0.0
5,2611606056,Santana,6,1013.0,11677.45,8500.0,0.0,0.0
6,2611606058,Monteiro,9,2342.0,11028.80,10001.0,0.0,0.0
7,2611606015,Aflitos,10,1998.0,11008.62,8000.0,0.0,0.0
8,2611606017,Rosarinho,12,2818.0,10288.50,7225.0,0.0,0.0
9,2611606016,Espinheiro,19,4374.0,9868.23,8000.0,0.0,0.0


## 7 – Nota de 0 a 10 por bairro (normalização min-max)

A nota é calculada pela fórmula:

$$\text{nota} = \frac{\text{renda\_media} - \text{renda\_min}}{\text{renda\_max} - \text{renda\_min}} \times 10$$

Bairro com menor renda recebe **0,00** e o de maior renda recebe **10,00**.

In [26]:
renda_min = agg['renda_media'].min()
renda_max = agg['renda_media'].max()

agg['nota_dimensao'] = ((agg['renda_media'] - renda_min) / (renda_max - renda_min) * 10).round(2)

print(f'Renda mínima : R$ {renda_min:,.2f}  → nota 0,00')
print(f'Renda máxima : R$ {renda_max:,.2f}  → nota 10,00')
print(f'Renda média  : R$ {agg["renda_media"].mean():,.2f}  → nota {((agg["renda_media"].mean() - renda_min) / (renda_max - renda_min) * 10):.2f}')

notas_exibir = (
    agg[['NM_BAIRRO', 'renda_media', 'nota_dimensao', 'pct_pobreza']]
    .sort_values('nota_dimensao', ascending=False)
    .reset_index(drop=True)
)
notas_exibir.index += 1

display(
    notas_exibir.style
    .format({
        'renda_media':    'R$ {:,.2f}',
        'nota_dimensao':  '{:.2f}',
        'pct_pobreza':    '{:.1f}%',
    })
    .background_gradient(subset=['nota_dimensao'], cmap='RdYlGn')
    .set_caption('Nota de Renda (0–10) por Bairro — Recife')
)

Renda mínima : R$ 1,116.60  → nota 0,00
Renda máxima : R$ 16,337.43  → nota 10,00
Renda média  : R$ 4,068.80  → nota 1.94


,NM_BAIRRO,renda_media,nota_dimensao,pct_pobreza
1,Jaqueira,"R$ 16,337.43",10.00,0.0%
2,Casa Forte,"R$ 14,308.91",8.67,0.0%
3,Parnamirim,"R$ 13,405.17",8.07,2.4%
4,Poço,"R$ 12,168.53",7.26,0.0%
5,Graças,"R$ 11,761.22",6.99,0.0%
6,Santana,"R$ 11,677.45",6.94,0.0%
7,Monteiro,"R$ 11,028.80",6.51,0.0%
8,Aflitos,"R$ 11,008.62",6.50,0.0%
9,Rosarinho,"R$ 10,288.50",6.03,0.0%
10,Tamarineira,"R$ 9,867.05",5.75,0.0%


In [27]:
fig3 = px.bar(
    notas_exibir.sort_values('nota_dimensao'),
    x='nota_dimensao',
    y='NM_BAIRRO',
    orientation='h',
    color='nota_dimensao',
    color_continuous_scale='RdYlGn',
    range_color=[0, 10],
    text='nota_dimensao',
    hover_data={'renda_media': ':,.2f', 'pct_pobreza': ':.1f'},
    labels={
        'nota_dimensao': 'Nota (0–10)',
        'NM_BAIRRO':     'Bairro',
        'renda_media':   'Renda Média (R$)',
        'pct_pobreza':   '% em Pobreza',
    },
    title='Índice de Renda por Bairro — Recife (Censo 2022)',
    height=1800,
)

fig3.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig3.update_layout(
    margin=dict(l=10, r=60, t=50, b=10),
    showlegend=False,
    coloraxis_showscale=False,
    xaxis=dict(range=[0, 11], title='Nota (0–10)'),
)

fig3.show()

## 8 – Salvar notas_renda.csv

In [28]:
OUTPUT_NOTAS = os.path.join(ROOT, 'data', 'processed', 'notas_renda.csv')


def format_brl(valor: float) -> str:
    """Formata valor em reais no padrão brasileiro: R$ 1.234,56"""
    return f"R$ {valor:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')


notas_renda = pd.DataFrame({
    'bairro':         agg['NM_BAIRRO'],
    'nota_dimensao':  agg['nota_dimensao'],
    'dado_principal': agg['renda_media'].apply(format_brl),
})

notas_renda = notas_renda.sort_values('nota_dimensao', ascending=False).reset_index(drop=True)

notas_renda.to_csv(OUTPUT_NOTAS, index=False, encoding='utf-8-sig')

print(f'Arquivo salvo: {OUTPUT_NOTAS}')
print(f'Total de bairros: {len(notas_renda)}')
display(notas_renda.head(10))
print('...')
display(notas_renda.tail(10))

Arquivo salvo: c:\Users\rodri\OneDrive\Documents\ProjetosPessoais\IVU-RECIFE\data\processed\notas_renda.csv
Total de bairros: 94


,bairro,nota_dimensao,dado_principal
0,Jaqueira,10.00,"R$ 16.337,43"
1,Casa Forte,8.67,"R$ 14.308,91"
2,Parnamirim,8.07,"R$ 13.405,17"
3,Poço,7.26,"R$ 12.168,53"
4,Graças,6.99,"R$ 11.761,22"
5,Santana,6.94,"R$ 11.677,45"
6,Monteiro,6.51,"R$ 11.028,80"
7,Aflitos,6.50,"R$ 11.008,62"
8,Rosarinho,6.03,"R$ 10.288,50"
9,Tamarineira,5.75,"R$ 9.867,05"


...


,bairro,nota_dimensao,dado_principal
84,Brejo de Beberibe,0.20,"R$ 1.427,46"
85,Linha do Tiro,0.20,"R$ 1.426,11"
86,Dois Unidos,0.17,"R$ 1.371,21"
87,Córrego do Jenipapo,0.16,"R$ 1.364,08"
88,Alto Santa Terezinha,0.15,"R$ 1.349,52"
89,Caçote,0.09,"R$ 1.246,28"
90,Passarinho,0.05,"R$ 1.193,83"
91,Peixinhos,0.05,"R$ 1.195,42"
92,Ilha Joana Bezerra,0.01,"R$ 1.134,42"
93,Recife,0.00,"R$ 1.116,60"
